# components/stats_panel

> Stats summary and verify selection button

In [ ]:
#| default_exp components.stats_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional

from fasthtml.common import Div, Span, Button, P

from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_colors, btn_styles
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, text_align
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, flex_wrap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V10 P2 content_panel, V11 icon-size roles)
from cjm_fasthtml_design_system.panels import panels
from cjm_fasthtml_design_system.icons import icons

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile, ExtractionResult
from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds

## Stats Panel

Shows file counts and the verify selection button. After verification, shows a success message.

In [ ]:
#| export
_STATS_CONTAINER_CLS = combine_classes(
    w.full, m.t(4),
    panels.content_panel,
)


def render_stats_content(
    selected_files: List[SelectedFile],  # Current selection
    urls: SourceSelectUrls,  # URL bundle
    extraction_results: Optional[Dict[str, ExtractionResult]] = None,  # Extraction results
    verified: bool = False,  # Whether selection is verified
    oob: bool = False,  # Whether to render as OOB outerHTML swap
) -> Div:  # Stats content (inner content only, no container chrome)
    """Render stats panel inner content for OOB updates."""
    extraction_results = extraction_results or {}
    audio_count = sum(1 for f in selected_files if f.get("file_type") == "audio")
    video_count = sum(1 for f in selected_files if f.get("file_type") == "video")
    total = len(selected_files)

    # Count extraction statuses
    videos_needing_extraction = 0
    videos_extracted = 0
    videos_with_error = 0
    for f in selected_files:
        if f.get("file_type") == "video":
            result = extraction_results.get(f["path"])
            if result and result.get("status") == "complete":
                videos_extracted += 1
            elif result and result.get("status") == "error":
                videos_with_error += 1
            else:
                videos_needing_extraction += 1

    oob_attr = {"hx_swap_oob": "outerHTML"} if oob else {}

    # Empty state
    if total == 0:
        return Div(
            P(
                "Select audio or video files to begin",
                cls=combine_classes(text_dui.base_content.opacity(40), font_size.sm, text_align.center)
            ),
            id=SourceSelectHtmlIds.STATS_CONTENT,
            cls=str(p(4)),
            **oob_attr,
        )

    # Stats text
    parts = [f"{total} file{'s' if total != 1 else ''} selected"]
    if audio_count:
        parts.append(f"{audio_count} audio")
    if video_count:
        video_detail = f"{video_count} video"
        if videos_needing_extraction > 0 and not verified:
            video_detail += f" (needs extraction)"
        elif videos_extracted == video_count:
            video_detail += " (extracted)"
        parts.append(video_detail)
    stats_text = " · ".join(parts)

    # Verified state
    if verified:
        return Div(
            Div(
                lucide_icon("circle-check", size=icons.status_inline, cls=str(text_dui.success)),
                Span(
                    f"Selection verified · {total} audio source{'s' if total != 1 else ''} ready",
                    cls=combine_classes(font_size.sm, text_dui.success, m.l(2))
                ),
                cls=combine_classes(str(flex_display), items.center)
            ),
            id=SourceSelectHtmlIds.STATS_CONTENT,
            cls=combine_classes(str(flex_display), justify.center, items.center, p(4)),
            **oob_attr,
        )

    # Unverified state with button
    return Div(
        Span(stats_text, cls=combine_classes(font_size.sm, text_dui.base_content)),
        Button(
            "Verify Selection",
            cls=combine_classes(btn, btn_colors.primary, btn_sizes.sm),
            hx_post=urls.verify,
            hx_swap="none",
        ),
        id=SourceSelectHtmlIds.STATS_CONTENT,
        cls=combine_classes(
            str(flex_display), justify.between, items.center,
            flex_wrap.wrap, gap(4), p(4)
        ),
        **oob_attr,
    )


def render_stats_panel(
    selected_files: List[SelectedFile],  # Current selection
    urls: SourceSelectUrls,  # URL bundle
    extraction_results: Optional[Dict[str, ExtractionResult]] = None,  # Extraction results
    verified: bool = False,  # Whether selection is verified
) -> Div:  # Full stats panel with container chrome
    """Render the full stats panel (container + content) for initial render."""
    content = render_stats_content(selected_files, urls, extraction_results, verified)
    return Div(
        content,
        id=SourceSelectHtmlIds.STATS_PANEL,
        cls=_STATS_CONTAINER_CLS,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()